## Volatility Targeting


Harvey, C. R. (with Rattray, S., & Hemert, O. van) have one section to discuss this topic in their book Strategic Risk Management: Designing Portfolios and Managing Risk. John Wiley & Sons, Incorporated.

This is heavily inspired what they implemented and tried in their book.


In short, this is to target to keep the volatility of the whole portfolio at a specific number or range instead of keep a fixed allocation in number of different assets.
For example, if your target volatility is 15%, if the current volatility is at 30%, you should sell your risky asset to lower the volatilty, vice versa. The goal is to manage the volatility to manage the risk.


Let use QQQ, BIL as an example


In [ ]:
%matplotlib inline

In [ ]:
from tiportfolio.helpers.data import Alpaca
import dotenv

dotenv.load_dotenv()

import os
alpaca = Alpaca(os.environ['ALPACA_API_KEY'], os.environ['ALPACA_API_SECRET'])

In [ ]:
from tiportfolio.helpers.data import YFinance
import pandas as pd
from datetime import datetime

# Initialize data sources
yf = YFinance()

# Define symbols and date range
symbols = ["QQQ", "BIL"]
start_date = "2020-01-01"
end_date = "2024-12-31"

# Fetch stock data from Alpaca
stock_data_df = alpaca.query(symbols, start_date, end_date)

# Fetch VIX data from Yahoo Finance
vix_df = yf.query(["^VIX"], start_date, end_date)
vix_series = vix_df.set_index('date')['close']

# Prepare dataframes for each strategy
qqq_prices = stock_data_df[stock_data_df['symbol'] == 'QQQ'].copy()
bil_prices = stock_data_df[stock_data_df['symbol'] == 'BIL'].copy()

# Ensure we have common dates and aligned timestamps (Midnight)
vix_series.index = pd.to_datetime(vix_series.index).tz_localize(None)
qqq_prices['date'] = pd.to_datetime(qqq_prices['date']).dt.tz_localize(None)
bil_prices['date'] = pd.to_datetime(bil_prices['date']).dt.tz_localize(None)

common_dates = set(qqq_prices['date']).intersection(set(bil_prices['date'])).intersection(set(vix_series.index))
common_dates = sorted(list(common_dates))

qqq_prices = qqq_prices[qqq_prices['date'].isin(common_dates)].set_index('date')
bil_prices = bil_prices[bil_prices['date'].isin(common_dates)].set_index('date')
vix_series = vix_series[vix_series.index.isin(common_dates)]

In [ ]:
from tiportfolio.strategy_library.trading.long_hold import LongHold
from tiportfolio.portfolio.allocation.vix_targeting import VixTargetingAllocation
from tiportfolio.portfolio.allocation.allocation import CASH_STRATEGY_NAME

# Create strategies
qqq_strategy = LongHold("QQQ", qqq_prices)
bil_strategy = LongHold("BIL", bil_prices)

# Configuration for the portfolio
config = {
    "initial_capital": 100000.0,
    "commission": 0.0001,  # 1bp
    "slippage": 0.0001,    # 1bp
    "risk_free_rate": 0.02,
    "market_name": "NYSE"
}

# 1. VIX Threshold-based Targeting Allocation
# We use QQQ (Flag=0.5) and BIL (Flag=0.1) as strategies.
# When rebalancing, they will be weighted such that W_i * F_i * VIX = Target_Vol / N
vix_targeting = VixTargetingAllocation(
    config=config,
    strategies=[qqq_strategy, bil_strategy],
    vix_data=vix_series,
    volatility_flags=[0.5, 0.1],
    target_vol=15.0,
    target_range=(15.0, 30.0),
    max_leverage=1.0
)

vix_targeting.walk_forward()
vix_targeting.evaluate()
metrics_vix = vix_targeting.get_performance_metrics(plot=True, fig_size=(15, 7))
print("VIX Threshold Targeting Metrics:")
for k, v in metrics_vix.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
# Benchmark: Fixed 60/40 Allocation (Rebalanced monthly)
from tiportfolio.strategy_library.allocation.fix_ratio import FixRatioFrequencyBasedAllocation
from tiportfolio.portfolio.allocation.frequency_based_allocation import RebalanceFrequency

benchmark_allocation = FixRatioFrequencyBasedAllocation(
    config=config,
    strategies=[qqq_strategy, bil_strategy],
    allocation_ratio_list=[0.6, 0.4],
    rebalance_frequency=RebalanceFrequency.end_of_month,
    hour=0, minute=0 # Align with data timestamps
)

benchmark_allocation.walk_forward()
benchmark_allocation.evaluate()
metrics_benchmark = benchmark_allocation.get_performance_metrics(plot=True, fig_size=(15, 7))
print("Benchmark (60/40) Metrics:")
for k, v in metrics_benchmark.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
import matplotlib.pyplot as plt

print("\nComparison Summary:")
print(f"{'Metric':<20} | {'VIX Targeting':<15} | {'Benchmark (60/40)':<15}")
print("-" * 65)
for k in metrics_vix.keys():
    print(f"{k:<20} | {metrics_vix[k]:<15.4f} | {metrics_benchmark[k]:<15.4f}")

In [ ]:
# Analysis of rebalance events
rebalance_dates = sorted(set([d for (d, s) in vix_targeting.strategy_ratio_map.keys()]))
print(f"Total rebalance events: {len(rebalance_dates)}")
if rebalance_dates:
    print(f"First rebalance: {rebalance_dates[0]}")
    print(f"Last rebalance: {rebalance_dates[-1]}")

# Check VIX distribution at rebalance times
vix_at_rebalance = vix_series.loc[rebalance_dates]
print("\nVIX at rebalance times:")
print(vix_at_rebalance.describe())